# Prompt 1: Spark Architecture Fundamentals
**Databricks Certified Associate Developer for Apache Spark — Topic 1 (20%)**

---

## Overview

This notebook covers the core Apache Spark architecture concepts required for the exam:

| Concept | Exam Weight |
|---|---|
| Driver vs Executor nodes | High |
| Physical execution hierarchy | High |
| Logical execution hierarchy | High |
| DAGScheduler & TaskScheduler | Medium |
| Cluster Manager role | Medium |
| SparkContext & SparkSession | High |

---
## 1. Architecture Diagram

```
┌─────────────────────────────────────────────────────────────────┐
│                        CLUSTER MANAGER                          │
│          (Standalone / YARN / Kubernetes / Mesos)               │
│   Grants worker nodes/containers to Spark on request           │
└───────────────────────────┬─────────────────────────────────────┘
                            │ resource negotiation
┌───────────────────────────▼─────────────────────────────────────┐
│                         DRIVER NODE                             │
│  ┌─────────────────┐   ┌──────────────┐   ┌────────────────┐  │
│  │  SparkContext / │   │ DAGScheduler │   │ TaskScheduler  │  │
│  │  SparkSession   │──▶│              │──▶│                │  │
│  └─────────────────┘   │ Logical Plan │   │ Submits Tasks  │  │
│                         │ → DAG of    │   │ to Executors   │  │
│                         │   Stages    │   └────────────────┘  │
│                         └──────────────┘                        │
└───────────────────────────┬─────────────────────────────────────┘
           ┌────────────────┼────────────────┐
           │                │                │
┌──────────▼──┐   ┌─────────▼───┐   ┌───────▼─────┐
│  EXECUTOR 1  │   │  EXECUTOR 2  │   │  EXECUTOR 3  │
│  (Worker Node│   │  (Worker Node│   │  (Worker Node│
│   JVM)       │   │   JVM)       │   │   JVM)       │
│  ┌─────────┐ │   │  ┌─────────┐ │   │  ┌─────────┐ │
│  │ Task 1  │ │   │  │ Task 3  │ │   │  │ Task 5  │ │
│  │ Task 2  │ │   │  │ Task 4  │ │   │  │ Task 6  │ │
│  └─────────┘ │   │  └─────────┘ │   │  └─────────┘ │
│  Cache/Memory│   │  Cache/Memory│   │  Cache/Memory│
└─────────────┘   └─────────────┘   └─────────────┘
```

**Key Points:**
- The **Driver** is a single JVM process that hosts `SparkContext`/`SparkSession`
- **Executors** are JVM processes running on worker nodes — they execute Tasks and store cached data
- The **Cluster Manager** is external; Spark requests resources from it but does not manage the cluster itself
- **Tasks** are the smallest unit of work — one Task processes one Partition

---
## 2. Physical Execution Hierarchy

```
Driver
  └── Executor (JVM on worker node)
        └── Task (1 per Partition — smallest unit of work)
```

| Component | What it is | Where it runs |
|---|---|---|
| Driver | JVM hosting SparkContext; orchestrates the job | Submitting machine (client) or cluster node (cluster deploy mode) |
| Executor | JVM launched on a worker node; runs Tasks and caches data | Worker nodes |
| Task | Single unit of work; processes one Partition | Inside an Executor |

> **Exam tip**: Each Task processes exactly one Partition. More partitions = more parallelism (up to the number of available executor cores).

---
## 3. Logical Execution Hierarchy

```
Action  (e.g., df.count(), df.collect(), df.write())
  └── Job  (one Job per Action)
        └── Stage  (split at shuffle boundaries — wide transformations)
              └── Task  (one per Partition within the Stage)
```

### How it works step by step:

1. You call an **Action** → Spark submits a **Job** to the DAGScheduler
2. The **DAGScheduler** converts the logical plan into a **DAG of Stages**, splitting at *wide transformation* (shuffle) boundaries
3. Each **Stage** is either a `ShuffleMapStage` (intermediate) or a `ResultStage` (final)
4. The **TaskScheduler** receives each Stage and submits its **Tasks** to available Executors
5. Each **Task** runs on one Partition in one Executor

### Narrow vs Wide Transformations:

| Type | Definition | Examples | Creates new Stage? |
|---|---|---|---|
| **Narrow** | Each input partition contributes to exactly one output partition | `filter`, `select`, `map`, `withColumn` | No |
| **Wide** | Input partitions contribute to multiple output partitions (requires shuffle) | `groupBy`, `join`, `orderBy`, `distinct`, `repartition` | **Yes** |

> **Exam tip**: Wide transformations = shuffle = new Stage boundary. Minimizing wide transformations reduces shuffles and improves performance.

---
## 4. DAGScheduler and TaskScheduler

### DAGScheduler
- Receives the **physical plan** from the optimizer
- Converts it into a **DAG (Directed Acyclic Graph) of Stages**
- Splits at wide transformation (shuffle) boundaries
- Two Stage types:
  - `ShuffleMapStage`: intermediate stage; writes shuffle output to disk
  - `ResultStage`: final stage; returns result to the driver
- Handles stage-level fault recovery (re-submits failed stages)

### TaskScheduler
- Receives completed Stages from the DAGScheduler
- Submits individual **Tasks** to available Executors via the `SchedulerBackend`
- Applies **data locality** preference (tries to schedule Tasks on nodes holding the data)
- Handles task-level retries on failure (default: 4 retries per task)

```
User Code
   │
   ▼ Action called
DAGScheduler ──► splits into Stages ──► ShuffleMapStage(s) + ResultStage
   │
   ▼ submits Stage
TaskScheduler ──► submits Tasks ──► SchedulerBackend ──► Executors
```

---
## 5. SparkContext and SparkSession

| Component | Introduced | Purpose |
|---|---|---|
| `SparkContext` | Spark 1.x | Entry point to the Spark cluster; manages RDDs | 
| `SparkSession` | Spark 2.0 | Unified entry point for DataFrame/SQL APIs; wraps SparkContext |

- `SparkSession` is the **preferred modern entry point** — use this for all DataFrame and Spark SQL work
- Access the underlying `SparkContext` via `spark.sparkContext`
- In Databricks notebooks, `spark` is pre-created automatically
- Only **one SparkContext** can be active per JVM; `SparkSession` can have multiple per context

> **Exam tip**: `SparkSession` = unified entry point for DataFrames + SQL. `SparkContext` = RDD-level API. When using Spark Connect, `SparkContext` is **not available** on the client.

In [ ]:
# Creating a SparkSession (standard pattern)
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SparkArchitectureDemo")
    .master("local[*]")  # local mode: use all cores
    .getOrCreate()
)

# Access the SparkContext from the session
sc = spark.sparkContext

print(f"Spark version: {spark.version}")
print(f"App name: {sc.appName}")
print(f"Master: {sc.master}")

---
## 6. Practical Examples — Observing the Hierarchy

In [ ]:
# Example 1: Observe how Actions trigger Jobs and Stages
# Each Action = 1 Job; shuffle boundaries = Stage splits

from pyspark.sql.functions import col

# Create sample data
data = [("Alice", "Engineering", 95000),
        ("Bob",   "Marketing",   72000),
        ("Carol", "Engineering", 88000),
        ("Dave",  "Marketing",   65000),
        ("Eve",   "Engineering", 102000)]

df = spark.createDataFrame(data, ["name", "department", "salary"])

# These are narrow transformations — no new Stage
filtered = df.filter(col("salary") > 70000)   # narrow
selected = filtered.select("name", "department", "salary")  # narrow

# groupBy triggers a shuffle — creates a Stage boundary
result = selected.groupBy("department").avg("salary")  # wide

# This Action triggers 1 Job with (at least) 2 Stages:
# Stage 0: filter + select (narrow)
# Stage 1: groupBy shuffle + avg (wide)
result.show()
# Check Spark UI → Jobs tab to see the 1 job
# Click the job → Stages tab to see the 2 stages

In [ ]:
# Example 2: Inspect partition count — relates to Task count
# Each Task processes 1 Partition

print(f"Partition count (= number of Tasks in this stage): {df.rdd.getNumPartitions()}")

# Repartition to control parallelism
df_repartitioned = df.repartition(4)
print(f"After repartition(4): {df_repartitioned.rdd.getNumPartitions()} partitions")

# When this runs, 4 Tasks will be submitted (one per partition)

In [ ]:
# Example 3: Multiple actions = multiple jobs
# Each call to an action triggers a new Job

count = df.count()        # Job 1 triggered
first = df.first()        # Job 2 triggered
df.show()                 # Job 3 triggered

print(f"Row count: {count}")
print(f"First row: {first}")
# Check Spark UI → Jobs tab — you will see 3 separate jobs

In [ ]:
# Example 4: Reading Cluster Manager configuration
# SparkContext exposes master and app info

sc = spark.sparkContext
print(f"Master (cluster manager): {sc.master}")
# local[*]  → Local mode (no external cluster manager)
# spark://  → Standalone cluster manager
# yarn      → YARN cluster manager
# k8s://    → Kubernetes cluster manager

print(f"Default parallelism (cores available): {sc.defaultParallelism}")
print(f"Default min partitions: {sc.defaultMinPartitions}")

---
## 7. Real-World Scenarios

<details>
<summary>Scenario 1: Why did my job create 3 stages instead of 1?</summary>

**Situation**: A data engineer writes a pipeline that reads a CSV, filters rows, joins with a lookup table, groups by region, and writes to Parquet. They notice 3 stages in the Spark UI and want to understand why.

**Explanation**: Two wide transformations create stage boundaries:
1. The `join` shuffles data so matching keys land on the same executor → **Stage boundary 1**
2. The `groupBy` shuffles again to co-locate same-region rows → **Stage boundary 2**
Result: Stage 0 (read + filter), Stage 1 (join shuffle), Stage 2 (groupBy + write)

```python
from pyspark.sql.functions import broadcast

sales = spark.read.csv("/data/sales.csv", header=True)
regions = spark.createDataFrame([("US", "North America"), ("UK", "Europe")], ["code", "region_name"])

# Stage 0: narrow (read + filter)
filtered = sales.filter(sales.amount > 0)

# Stage 1 boundary: wide (join shuffle)
joined = filtered.join(regions, on="code", how="left")

# Stage 2 boundary: wide (groupBy shuffle)
result = joined.groupBy("region_name").sum("amount")

result.write.parquet("/output/region_totals")
```

**Expected outcome**: Spark UI shows 3 stages. To reduce to 2, use `broadcast(regions)` since it's small — this eliminates the join shuffle.

**Exam sub-topic**: Logical execution hierarchy (Job → Stage → Task); wide vs narrow transformations
</details>

<details>
<summary>Scenario 2: One executor is doing all the work — why?</summary>

**Situation**: A Spark job with 10 executors completes slowly. The Spark UI Tasks tab shows 9 executors idle while 1 executor processes all tasks. The engineer suspects the partition count is the problem.

**Diagnosis**: The DataFrame has only 1 partition, so only 1 Task exists → only 1 Executor is used regardless of cluster size.

```python
# Problem: single file → 1 partition → 1 task → 1 executor used
df = spark.read.csv("/data/one_big_file.csv", header=True)
print(df.rdd.getNumPartitions())  # Output: 1

# Fix: repartition to match available parallelism
df = df.repartition(40)  # 10 executors × 4 cores each
print(df.rdd.getNumPartitions())  # Output: 40
```

**Expected outcome**: After repartition, 40 Tasks run in parallel across all executors. Job completes ~40× faster.

**Exam sub-topic**: Physical execution hierarchy (Executor → Task → Partition relationship)
</details>

<details>
<summary>Scenario 3: How does the cluster manager interact with Spark on YARN?</summary>

**Situation**: A team deploys Spark on a Hadoop cluster using YARN. A new engineer asks why Spark doesn't directly control the worker nodes and what YARN's role is.

**Explanation**:
1. Spark submits a resource request to the **YARN ResourceManager** (the Cluster Manager)
2. YARN's ResourceManager negotiates containers on worker nodes with NodeManagers
3. YARN launches the Spark **ApplicationMaster** (acts as the Spark Driver in cluster mode)
4. The ApplicationMaster requests Executor containers from YARN
5. Once Executors are running, Spark's TaskScheduler sends Tasks directly to them

```bash
# Submit to YARN cluster manager, driver runs inside cluster
spark-submit \
  --master yarn \
  --deploy-mode cluster \
  --executor-memory 4g \
  --executor-cores 2 \
  --num-executors 10 \
  my_spark_app.py
```

**Expected outcome**: YARN allocates 10 executor containers × (4g RAM + 2 cores). Spark manages task scheduling within those containers.

**Exam sub-topic**: Cluster Manager role; Spark does not manage cluster resources itself
</details>

<details>
<summary>Scenario 4: Multiple SparkSessions in the same application</summary>

**Situation**: A developer wants to run two independent Spark pipelines in the same Python process with different configurations (e.g., different databases). They wonder if they need two separate JVM processes.

**Explanation**: One `SparkContext` per JVM (enforced by Spark), but multiple `SparkSession` instances can share the same `SparkContext`. Each session can have its own SQL configuration (e.g., `spark.sql.shuffle.partitions`) without affecting the other.

```python
# Primary session
spark1 = SparkSession.builder.appName("Pipeline1").getOrCreate()

# Second session shares the same SparkContext
spark2 = spark1.newSession()
spark2.conf.set("spark.sql.shuffle.partitions", "50")

print(spark1.conf.get("spark.sql.shuffle.partitions"))  # 200 (default)
print(spark2.conf.get("spark.sql.shuffle.partitions"))  # 50

# Both share the same underlying SparkContext
print(spark1.sparkContext is spark2.sparkContext)  # True
```

**Expected outcome**: Two independent sessions with isolated SQL configs, sharing one SparkContext and one cluster connection.

**Exam sub-topic**: SparkContext vs SparkSession; one SparkContext per JVM rule
</details>

<details>
<summary>Scenario 5: Tracing a slow task back to a skewed partition</summary>

**Situation**: A Spark job has 200 tasks but runs for 20 minutes. Checking the Spark UI Tasks tab, 199 tasks finish in 30 seconds but one task runs for 19+ minutes. The engineer needs to diagnose and fix this.

**Diagnosis**: This is **data skew** — one partition contains far more data than others (e.g., one `groupBy` key accounts for 80% of rows). The TaskScheduler assigns one Task per Partition; if one Partition is huge, its Task becomes a straggler.

```python
# Diagnose: check partition sizes
from pyspark.sql.functions import spark_partition_id

df.withColumn("pid", spark_partition_id()) \
  .groupBy("pid").count() \
  .orderBy("count", ascending=False) \
  .show(10)
# If one pid has 10M rows and others have 50K, that confirms skew

# Fix option 1: Enable Adaptive Query Execution (AQE)
spark.conf.set("spark.sql.adaptive.enabled", "true")  # Spark 3.x default: true

# Fix option 2: Salt the skewed key
from pyspark.sql.functions import concat, col, lit, floor, rand
import math

SALT_FACTOR = 10
df_salted = df.withColumn("salted_key", concat(col("skewed_col"), lit("_"), (rand() * SALT_FACTOR).cast("int").cast("string")))
result = df_salted.groupBy("salted_key").count()
# Then strip salt and re-aggregate
```

**Expected outcome**: AQE automatically splits the large partition into smaller tasks. Manual salting distributes the skewed key across multiple partitions.

**Exam sub-topic**: Physical execution hierarchy (Task → Partition mapping); data skew identification
</details>

---
## 8. Quick Reference Summary

| Concept | Key Fact |
|---|---|
| Driver | Single JVM; hosts SparkContext; orchestrates jobs |
| Executor | JVM on worker node; runs Tasks; caches data |
| Task | Smallest work unit; processes 1 Partition |
| Action | Triggers a Job (e.g., `count()`, `show()`, `write()`) |
| Job | One per Action; divided into Stages |
| Stage | Split at shuffle (wide transformation) boundaries |
| DAGScheduler | Converts plan to DAG of Stages |
| TaskScheduler | Submits Tasks to Executors |
| Cluster Manager | External resource allocator (YARN/K8s/Standalone) |
| SparkSession | Preferred entry point (Spark 2+); wraps SparkContext |
| SparkContext | RDD-level API; one per JVM; not available via Spark Connect |

---
## 9. Exam Practice Questions

**Q1**: An Action is called on a DataFrame that was built using `filter()` → `join()` → `groupBy()`. How many Stage boundaries will there be at minimum?

<details><summary>Answer</summary>
**2 Stage boundaries** (minimum). Both `join` and `groupBy` are wide transformations that require shuffles, creating 2 stage boundaries and 3 stages total. `filter()` is narrow and does not create a boundary.
</details>

**Q2**: You have 100 partitions and 10 executor cores. How many Tasks will be active simultaneously at peak?

<details><summary>Answer</summary>
**10 Tasks simultaneously** — one per available executor core. The 100 tasks will be processed in ~10 waves of 10 tasks each.
</details>

**Q3**: What is the difference between `SparkContext` and `SparkSession`?

<details><summary>Answer</summary>
`SparkContext` is the Spark 1.x entry point for the RDD API; one per JVM. `SparkSession` is the Spark 2.0+ unified entry point for the DataFrame and SQL APIs; multiple sessions can share one SparkContext. Access SparkContext via `spark.sparkContext`.
</details>

**Q4**: Which component is responsible for splitting a Job into Stages?

<details><summary>Answer</summary>
The **DAGScheduler** converts the physical plan into a DAG of Stages, splitting at wide transformation (shuffle) boundaries.
</details>

**Q5**: In YARN cluster deploy mode, where does the Spark Driver run?

<details><summary>Answer</summary>
On a **worker node inside the cluster** (as the YARN ApplicationMaster). The client machine that submitted the job does not need to stay running.
</details>